# Tutorial: LLM Feature Gen Quickstart

Audience:
- People evaluating the library or reviewing the repository.

Prerequisites:
- Python 3.9+ and `pip install llm-feature-gen` or `pip install -e ".[dev]"`.
- A `.env` file for OpenAI, Azure OpenAI, or a local OpenAI-compatible endpoint.
- `ffmpeg` on your machine if you want to run the video examples with audio.

Learning goals:
- Run the full discovery -> generation workflow once end to end.
- Understand the two directory layouts the package expects.
- Switch from the default provider to `LocalProvider` when needed.


## Outline

1. Inspect the bundled sample folders
2. Run a minimal text quickstart
3. Switch to `LocalProvider`
4. Reuse the same pattern for images, tabular data, and videos
5. Review the most common pitfalls and output files


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

REPO_ROOT = Path.cwd()
SAMPLE_DIRS = [
    "discover_texts",
    "texts",
    "discover_images",
    "images",
    "discover_tabular",
    "tabular",
    "discover_videos",
    "videos",
]

def summarize_dir(path_str: str) -> dict[str, object]:
    path = REPO_ROOT / path_str
    if not path.exists():
        return {"path": path_str, "exists": False}
    children = sorted(p.name for p in path.iterdir())
    return {
        "path": path_str,
        "exists": True,
        "is_dir": path.is_dir(),
        "entries": children[:8],
        "entry_count": len(children),
    }

[summarize_dir(path_str) for path_str in SAMPLE_DIRS]


## Step 1 - Understand the two layouts

The package uses two different input layouts:

- Discovery reads a single file, a folder, or a list of raw inputs and proposes one shared feature schema by default because `as_set=True`.
- Generation expects a root folder with one subfolder per class, such as `texts/advisory_texts/` and `texts/formal_texts/`.

That distinction is the main thing to keep in mind when moving from discovery to generation.


In [ ]:
discovery_layout = sorted((REPO_ROOT / "discover_texts").glob("*"))
generation_layout = sorted((REPO_ROOT / "texts").iterdir())

print("Discovery files:")
for path in discovery_layout:
    print(" -", path.name)

print("\nGeneration class folders:")
for path in generation_layout:
    print(" -", path.name)


## Step 2 - Run the minimal text workflow

This is the fastest happy path in the repo because the sample text folders are already present.
The first call writes `outputs/discovered_text_features.json`. The second call reads that schema and writes one CSV per class plus an optional merged CSV.


In [ ]:
from llm_feature_gen.discover import discover_features_from_texts
from llm_feature_gen.generate import generate_features_from_texts

discovered = discover_features_from_texts("discover_texts")
print(json.dumps(discovered, indent=2, ensure_ascii=False))

csv_paths = generate_features_from_texts(
    root_folder="texts",
    discovered_features_path="outputs/discovered_text_features.json",
    merge_to_single_csv=True,
)
csv_paths


## Step 3 - Switch to `LocalProvider`

If you have an OpenAI-compatible local server running, you can pass `LocalProvider` explicitly.
Typical `.env` values are:

```env
LOCAL_OPENAI_BASE_URL=http://localhost:11434/v1
LOCAL_OPENAI_API_KEY=ollama
LOCAL_MODEL_TEXT=llama3
LOCAL_MODEL_VISION=llava
LOCAL_WHISPER_MODEL_SIZE=base
LOCAL_WHISPER_DEVICE=cpu
```

For local video runs, install `faster-whisper` if you want audio transcription. If you do not have it, set `use_audio=False`.


In [ ]:
from llm_feature_gen.providers.local_provider import LocalProvider
from llm_feature_gen.discover import discover_features_from_texts

local_provider = LocalProvider()

local_discovered = discover_features_from_texts(
    "discover_texts",
    provider=local_provider,
    output_filename="discovered_text_features_local.json",
)
local_discovered


## Step 4 - Apply the same pattern to other modalities

Images, tabular data, and videos follow the same two-step flow. The main differences are the input folder names and a couple of modality-specific arguments.

### Images

```python
from llm_feature_gen.discover import discover_features_from_images
from llm_feature_gen.generate import generate_features_from_images

image_features = discover_features_from_images("discover_images")
image_csvs = generate_features_from_images(
    root_folder="images",
    discovered_features_path="outputs/discovered_image_features.json",
)
```

### Tabular

```python
from llm_feature_gen.discover import discover_features_from_tabular
from llm_feature_gen.generate import generate_features_from_tabular

tabular_features = discover_features_from_tabular(
    "discover_tabular",
    text_column="text",
)
tabular_csvs = generate_features_from_tabular(
    root_folder="tabular",
    discovered_features_path="outputs/discovered_tabular_features.json",
    text_column="text",
)
```

### Videos

```python
from llm_feature_gen.discover import discover_features_from_videos
from llm_feature_gen.generate import generate_features_from_videos

video_features = discover_features_from_videos(
    "discover_videos",
    num_frames=5,
    use_audio=False,
)
video_csvs = generate_features_from_videos(
    root_folder="videos",
    discovered_features_path="outputs/discovered_video_features.json",
    use_audio=False,
)
```


## Pitfalls and outputs

- Discovery defaults to `as_set=True`, so a folder is treated as one comparative batch unless you override that behavior.
- Generation does not read a flat folder of files. It looks for class subfolders under `root_folder`.
- Tabular generation requires `text_column`.
- Video generation should point at `outputs/discovered_video_features.json`, not `outputs/features_discover_videos.json`.
- Generated CSVs include `raw_llm_output`, which is useful when you need to audit or debug model responses.


## Exercise

Pick one modality you care about most and adapt the minimal text workflow to it.
Before you run generation, check that the root folder really contains class subfolders instead of files directly at the top level.


In [ ]:
def list_classes(root_folder: str) -> list[str]:
    root = Path(root_folder)
    return sorted(path.name for path in root.iterdir() if path.is_dir())

list_classes("texts")
